# Generating cutflow spreadsheets
Analysis algorithms often print out the cutflow table to the terminal.

This notebook uses some simple tools to manipulate these tables and convert them into Pandas Dataframes (DFs) and Excel spreadsheets.

Extract cutflow tables from terminal outputs text, convert them into DataFrames and save them as spreadsheets.

**Contents:**
1. DF Manipulation
1. `make_spreadsheet()` - saves multiple DFs to a single spreadsheet.
2. Functions to convert strings to DFs for different formats:
    - `str_to_df_zdzd13TeV()` - ZdZd13TeV format
    - `str_to_df_zdzdPP()` - ZdZdPostProcessing format
3. Cut-name maps - dictionaries mapping the cut names from one format to another.
    - ZdZdPostProcessing : ZdZd13TeV
4. Functions to extract cutflow tables from text files (NEED TO IMPLEMENT)

# Cutflow manipulation
All the functions are defined below and for the most part you won't have to change them.

If you want to see how to use them there are examples for each function.

In [63]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

# asdg
print(np.NaN)

nan


In [66]:
cutflow_dir = '/Users/matt/Documents/work/0-PhD/particle/xxzx-analysis/code_ZdZd/cutflows/'
scott_cutflows_2025_10 = '2025-10-17_Scott-ZdZd-R3-cutflows.txt'
scott_cutflows = parse_cutflow_file(cutflow_dir + scott_cutflows_2025_10)
scott_cutflows

{'zd5_23a --- reco': '|                 |N4e |Wt4e   |Nem  |Wtem   |N4m  |Wt4m   |N all |Wt all |\n|*Precuts*                                                        |||||||||\n|weight           |    |       |     |       |     |       |22846 |0.0000 |\n|overlap          |    |       |     |       |     |       |22846 |0.0000 |\n|clean            |    |       |     |       |     |       |22846 |0.0000 |\n|pv               |    |       |     |       |     |       |22846 |0.0000 |\n|trigger          |    |       |     |       |     |       |19443 |0.0000 |\n|jetclean         |    |       |     |       |     |       |19443 |0.0000 |\n|*Quad selection*                                                 |||||||||\n|quad sfos        |    |       |     |       |     |       |19104 |0.0000 |\n|quad or          |    |       |     |       |     |       |19103 |0.0000 |\n|quad kin         |    |       |     |       |     |       |18378 |0.0000 |\n|quad trigmatch   |    |       |     |       |     |  

In [72]:
zd30_23a_truth = str_to_df_ZdZd13TeV(scott_cutflows['zd30_23a --- truth'])
zd30_23a_reco = str_to_df_ZdZd13TeV(scott_cutflows['zd30_23a --- reco'])
display(zd30_23a_reco)
print(len(zd30_23a_truth))

,Cut,N4e,Wt4e,Nem,Wtem,N4m,Wt4m,N all,Wt all
0,weight,,,,,,,60000,0.0000
1,overlap,,,,,,,60000,0.0000
2,clean,,,,,,,60000,0.0000
3,pv,,,,,,,60000,0.0000
4,trigger,,,,,,,51707,0.0000
5,jetclean,,,,,,,51255,0.0000
6,*Quad selection*,,,,,,,,
7,quad sfos,,,,,,,22993,0.0000
8,quad or,,,,,,,22987,0.0000
9,quad kin,,,,,,,22319,0.0000


26


In [73]:
# Print values of zd30_23a_reco['Cut'] that do not appear in zd30_23a_truth['Cut']
for cut in zd30_23a_reco['Cut']:
    if cut not in zd30_23a_truth['Cut'].values:
        print(cut) 

overlap
clean
pv
trigger
jetclean
*Quad selection*
quad or
quad trigmatch
quad dr
isol
e_id
impact


## 1. DF dict --> spreadsheet

In [4]:
def make_spreadsheet(outfile, table_dict):
    '''Take in a filename and a dictionary with format: {sheet_name: table_df}
    Save as an excel file with each table in its own sheet.'''
    with pd.ExcelWriter(outfile) as writer:
        for sheet_name, table_df in table_dict.items():
            df = table_df
            df.to_excel(writer, sheet_name=sheet_name, index=False)

# Example use:
# make_spreadsheet('~/Downloads/cutflow_tables_ZdZdPP_23a.xlsx', {'ZdZdPP_23a_scott': str_to_df(zd30_23a), })

## 2. Cutflow string --> DF

Each analysis code prints the cutflows in slightly different formats, with different cut names.

### 2.1 ZdZd13TeV (Scott's format)

Scott likes to send cutflow tables via email, where each cell is separated by a '|'. These functions convert the email text into spreadsheets.

In [23]:
def str_to_df_ZdZd13TeV(table_str):
    '''Input a cutflow table string, return as a DataFrame'''
    lines = table_str.strip().split('\n')
    data = []
    for line in lines[2:]:  # Skip the first two header lines
        parts = line.split('|')[1:-1]  # Split by '|' and ignore empty parts
        row = [part.strip() for part in parts]
        data.append(row)
    columns = ['Cut', 'N4e', 'Wt4e', 'Nem', 'Wtem', 'N4m', 'Wt4m', 'N all', 'Wt all']
    df = pd.DataFrame(data, columns=columns)
    return df

In [24]:
zd30_23a = '''|                 |N4e  |Wt4e   |Nem  |Wtem   |N4m  |Wt4m   |N all |Wt all |
|*Precuts*                                                         |||||||||
|weight           |     |       |     |       |     |       |60000 |0.0000 |
|overlap          |     |       |     |       |     |       |60000 |0.0000 |
|clean            |     |       |     |       |     |       |60000 |0.0000 |
|pv               |     |       |     |       |     |       |60000 |0.0000 |
|trigger          |     |       |     |       |     |       |51707 |0.0000 |
|jetclean         |     |       |     |       |     |       |51255 |0.0000 |
|*Quad selection*                                                  |||||||||
|quad sfos        |     |       |     |       |     |       |22993 |0.0000 |
|quad or          |     |       |     |       |     |       |22987 |0.0000 |
|quad kin         |     |       |     |       |     |       |22319 |0.0000 |
|quad trigmatch   |     |       |     |       |     |       |22193 |0.0000 |
|quad dr          |     |       |     |       |     |       |22161 |0.0000 |
|*Common*                                                          |||||||||
|isol             |3010 |0.0000 |8775 |0.0000 |6634 |0.0000 |18419 |0.0000 |
|e_id             |3010 |0.0000 |8775 |0.0000 |6634 |0.0000 |18419 |0.0000 |
|impact           |2990 |0.0000 |8594 |0.0000 |6443 |0.0000 |18027 |0.0000 |
|jpsi             |2990 |0.0000 |8593 |0.0000 |6442 |0.0000 |18025 |0.0000 |
|upsilon          |2986 |0.0000 |8592 |0.0000 |6423 |0.0000 |18001 |0.0000 |
|lowmveto         |2986 |0.0000 |8590 |0.0000 |6422 |0.0000 |17998 |0.0000 |
|*HM*                                                              |||||||||
|hwind            |2601 |0.0000 |7884 |0.0000 |6169 |0.0000 |16654 |0.0000 |
|zveto            |1817 |0.0000 |7884 |0.0000 |4165 |0.0000 |13866 |0.0000 |
|loose            |1810 |0.0000 |7876 |0.0000 |4133 |0.0000 |13819 |0.0000 |
|medium           |1802 |0.0000 |7848 |0.0000 |4128 |0.0000 |13778 |0.0000 |
|tight            |521  |0.0000 |2355 |0.0000 |1493 |0.0000 |4369  |0.0000 |
|*AS SR1*                                                          |||||||||
|SR1              |114  |0.0000 |323  |0.0000 |217  |0.0000 |654   |0.0000 |
|zveto            |108  |0.0000 |323  |0.0000 |200  |0.0000 |631   |0.0000 |
|loose            |90   |0.0000 |267  |0.0000 |161  |0.0000 |518   |0.0000 |
|medium           |69   |0.0000 |197  |0.0000 |116  |0.0000 |382   |0.0000 |
|tight            |39   |0.0000 |67   |0.0000 |44   |0.0000 |150   |0.0000 |
|*AS SR2*                                                          |||||||||
|SR2              |271  |0.0000 |383  |0.0000 |36   |0.0000 |690   |0.0000 |
|zveto            |244  |0.0000 |383  |0.0000 |34   |0.0000 |661   |0.0000 |
|loose            |242  |0.0000 |383  |0.0000 |33   |0.0000 |658   |0.0000 |
|medium           |224  |0.0000 |353  |0.0000 |33   |0.0000 |610   |0.0000 |
|tight            |182  |0.0000 |269  |0.0000 |33   |0.0000 |484   |0.0000 |'''

Test function

In [25]:
zd30_23a_df = str_to_df_ZdZd13TeV(zd30_23a)
display(zd30_23a_df)

,Cut,N4e,Wt4e,Nem,Wtem,N4m,Wt4m,N all,Wt all
0,weight,,,,,,,60000,0.0000
1,overlap,,,,,,,60000,0.0000
2,clean,,,,,,,60000,0.0000
3,pv,,,,,,,60000,0.0000
4,trigger,,,,,,,51707,0.0000
5,jetclean,,,,,,,51255,0.0000
6,*Quad selection*,,,,,,,,
7,quad sfos,,,,,,,22993,0.0000
8,quad or,,,,,,,22987,0.0000
9,quad kin,,,,,,,22319,0.0000


### 2.2 ZdZdPostProcessing format

In [10]:
def str_to_df_zdzdPP(cutflow_str):
    lines = [line.strip() for line in cutflow_str.strip().split('\n') if line.strip() and not line.strip().startswith('-')]
    
    # Parse header
    header_parts = re.findall(r'\|([^|]+)', lines[0])
    header = ['Cuts'] + [col.strip() for col in header_parts[1:]]
    
    # Parse data rows
    data = []
    for line in lines[1:]:
        cols = [col.strip() for col in re.findall(r'\|([^|]+)', line)]
        data.append(cols)
    
    df = pd.DataFrame(data, columns=header)
    
    # Split columns (except 'Cuts') into weights and events
    new_cols = {}
    new_cols['Cuts'] = df['Cuts']
    
    for col in df.columns[1:]:
        # Extract float (weight) and integer (events)
        weights = df[col].str.extract(r'([0-9.]+)\s*\(')[0].astype(float)
        events = df[col].str.extract(r'\(([0-9]+)\)')[0].astype(int)
        
        new_cols[f'{col}_weights'] = weights
        new_cols[f'{col}_events'] = events
    
    return pd.DataFrame(new_cols)

Test ZdZdPostProcessing example

In [ ]:
# Example
zdzdPP_23a_cutflow = '''|                |#==561511           |#==5615111        |#==5615112          |#==5615113        |
--------------------------------------------------------------------------------------------------
|All             |60000.000000 (60000)|0.000000 (0)      |0.000000 (0)        |0.000000 (0)      |
|Cleaning        |60000.000000 (60000)|0.000000 (0)      |0.000000 (0)        |0.000000 (0)      |
|PrimaryVertex   |60000.000000 (60000)|0.000000 (0)      |0.000000 (0)        |0.000000 (0)      |
|Trigger         |51707.000000 (51707)|0.000000 (0)      |0.000000 (0)        |0.000000 (0)      |
|SFOS            |0.000000 (0)        |3760.000000 (3760)|11016.000000 (11016)|8217.000000 (8217)|
|NotOverlaps     |0.000000 (0)        |3760.000000 (3760)|11010.000000 (11010)|8217.000000 (8217)|
|Kinematic       |0.000000 (0)        |3709.000000 (3709)|10732.000000 (10732)|7878.000000 (7878)|
|TriggerMatched  |0.000000 (0)        |3676.000000 (3676)|10675.000000 (10675)|7842.000000 (7842)|
|ElQuality       |0.000000 (0)        |3676.000000 (3676)|10675.000000 (10675)|7842.000000 (7842)|
|MuType          |0.000000 (0)        |3676.000000 (3676)|10674.000000 (10674)|7833.000000 (7833)|
|DeltaR          |0.000000 (0)        |3671.000000 (3671)|10653.000000 (10653)|7827.000000 (7827)|
|ElectronID      |0.000000 (0)        |3671.000000 (3671)|10653.000000 (10653)|7827.000000 (7827)|
|ImpactParameterZ|0.000000 (0)        |3651.000000 (3651)|10596.000000 (10596)|7803.000000 (7803)|
|ImpactParameterD|0.000000 (0)        |3651.000000 (3651)|10596.000000 (10596)|7803.000000 (7803)|
|ImpactParameter |0.000000 (0)        |3633.000000 (3633)|10401.000000 (10401)|7582.000000 (7582)|
|QVeto           |0.000000 (0)        |3628.000000 (3628)|10399.000000 (10399)|7556.000000 (7556)|
|LowMassVeto     |0.000000 (0)        |3625.000000 (3625)|10397.000000 (10397)|7555.000000 (7555)|
|HWindow         |0.000000 (0)        |3101.000000 (3101)|9416.000000 (9416)  |7133.000000 (7133)|
|ZVeto           |0.000000 (0)        |2168.000000 (2168)|9416.000000 (9416)  |4805.000000 (4805)|
|LooseSR         |0.000000 (0)        |2168.000000 (2168)|9416.000000 (9416)  |4805.000000 (4805)|
|MediumSR        |0.000000 (0)        |2066.000000 (2066)|9179.000000 (9179)  |4759.000000 (4759)|
--------------------------------------------------------------------------------------------------'''

In [28]:
zdzdPP_23a_df = str_to_df_zdzdPP(zdzdPP_23a_cutflow)
# display(zdzdPP_23a_df)

Save ``zdzdPP_23a_df`` to an excel file

In [12]:
# Save df2 to an excel file
# make_spreadsheet('~/Downloads/zdzdPP_cutflows.xlsx', {'zdzdPP_23a': zdzdPP_23a_df})

***
## 3. Cut-name maps
***

Here we save some dictionaries of string-string pairs which represent the mapping of cut names between to cutflow formats.

If one cut doesn't appear in the other cutflow, it gets mapped to `'N/A'`.

### 3.1 ZdZd13TeV -> ZdZdPostProcessing

In [14]:
# Get column names from zdzdPP_23a_df
zdzdPP_23a_df['Cuts'].tolist()

['All',
 'Cleaning',
 'PrimaryVertex',
 'Trigger',
 'SFOS',
 'NotOverlaps',
 'Kinematic',
 'TriggerMatched',
 'ElQuality',
 'MuType',
 'DeltaR',
 'ElectronID',
 'ImpactParameterZ',
 'ImpactParameterD',
 'ImpactParameter',
 'QVeto',
 'LowMassVeto',
 'HWindow',
 'ZVeto',
 'LooseSR',
 'MediumSR']

In [57]:
zdPP_to_zd13TeV = {
    'All': 'weight',
    'N/A': 'overlap',
    'Cleaning': 'clean',
    'PrimaryVertex': 'pv',
    'Trigger': 'trigger',
    'N/A': 'jetclean',
    'SFOS': 'quad sfos',
    'NotOverlaps': 'quad or',
    'Kinematic': 'quad kin',
    'TriggerMatched': 'quad trigmatch',
    'ElQuality': 'N/A',
    'MuType': 'N/A',
    'DeltaR': 'quad dr',
    'ElectronID': 'e_id',
    'ImpactParameterZ': 'N/A', #Need to compare how impact params are defined in each repo
    'ImpactParameterD': 'N/A',
    'ImpactParameter': 'N/A',
    'N/A': 'impact',
    'QVeto': 'jpsi, upsilon', #Need to check
    'LowMassVeto': 'lowmveto',
    'HWindow': 'hwind',
    'ZVeto': 'zveto',
    'LooseSR': 'loose',
    'MediumSR': 'medium',
    'N/A': 'tight',
    'N/A': 'SR1',
    'N/A': 'zveto',
    'N/A': 'loose',
    'N/A': 'medium',
    'N/A': 'tight',
    'N/A': 'SR2',
    'N/A': 'zveto',
    'N/A': 'loose',
    'N/A': 'medium',
    'N/A': 'tight',
}

zdPP_to_zd13TeV_list = [
    ['All', 'N/A', 'Cleaning', 'PrimaryVertex', 'Trigger', 'N/A', 'SFOS', 'NotOverlaps', 'Kinematic', 'TriggerMatched', 'ElQuality', 'MuType', 'DeltaR', 'N/A', 'ElectronID', 'ImpactParameterZ', 'ImpactParameterD', 'ImpactParameter', 'N/A', 'QVeto', 'LowMassVeto', 'HWindow', 'ZVeto', 'LooseSR', 'MediumSR', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A', 'N/A'],
    ['weight', 'overlap', 'clean', 'pv', 'trigger', 'jetclean', 'quad sfos', 'quad or', 'quad kin', 'quad trigmatch', 'N/A', 'N/A', 'quad dr', 'isol', 'e_id', 'N/A', 'N/A', 'N/A', 'impact', 'jpsi, upsilon', 'lowmveto', 'hwind', 'zveto', 'loose', 'medium', 'tight', 'SR1', 'zveto', 'loose', 'medium', 'tight', 'SR2', 'zveto', 'loose', 'medium', 'tight']
]

In [58]:
print(len(zdPP_to_zd13TeV_list[0]))
print(len(zdPP_to_zd13TeV_list[1]))

36
36


In [59]:
# Convert zdPP_to_zd13TeV_list into a dataframe with columns 'zdPP_cut' and 'zd13TeV_cut'
zdPP_to_zd13TeV_df = pd.DataFrame({
    'zdPP_cut': zdPP_to_zd13TeV_list[0],
    'zd13TeV_cut': zdPP_to_zd13TeV_list[1]
})

display(zdPP_to_zd13TeV_df)

,zdPP_cut,zd13TeV_cut
0,All,weight
1,N/A,overlap
2,Cleaning,clean
3,PrimaryVertex,pv
4,Trigger,trigger
5,N/A,jetclean
6,SFOS,quad sfos
7,NotOverlaps,quad or
8,Kinematic,quad kin
9,TriggerMatched,quad trigmatch


In [47]:
# Convert dictstring into two lists - one list contains all the keys, the other all the values
keys = re.findall(r'"([^"]+)":', dicstring)
values = re.findall(r':\s*"([^"]+)"', dicstring)
print(values)

['weight', 'overlap', 'clean', 'pv', 'trigger', 'jetclean', 'quad sfos', 'quad or', 'quad kin', 'quad trigmatch', 'N/A', 'N/A', 'quad dr', 'e_id', 'N/A', 'N/A', 'N/A', 'impact', 'jpsi, upsilon', 'lowmveto', 'hwind', 'zveto', 'loose', 'medium', 'tight', 'SR1', 'zveto', 'loose', 'medium', 'tight', 'SR2', 'zveto', 'loose', 'medium', 'tight']


In [32]:
# Get column names from zd30_23a_df
zd30_23a_df['Cut'].tolist()

['weight',
 'overlap',
 'clean',
 'pv',
 'trigger',
 'jetclean',
 '*Quad selection*',
 'quad sfos',
 'quad or',
 'quad kin',
 'quad trigmatch',
 'quad dr',
 '*Common*',
 'isol',
 'e_id',
 'impact',
 'jpsi',
 'upsilon',
 'lowmveto',
 '*HM*',
 'hwind',
 'zveto',
 'loose',
 'medium',
 'tight',
 '*AS SR1*',
 'SR1',
 'zveto',
 'loose',
 'medium',
 'tight',
 '*AS SR2*',
 'SR2',
 'zveto',
 'loose',
 'medium',
 'tight']

In [62]:
# Print elements of zdzdPP_23a_df['Cut'] that do not appear in zdPP_to_zd13TeV.keys()
for cut in zd30_23a_df['Cut']:
    if cut not in zdPP_to_zd13TeV_df['zd13TeV_cut'].values:
        if '*' not in cut:  # Exclude section headers
            print(cut)

# Print elements of zdzdPP_23a_df['Cuts'] that do not appear in zdPP_to_zd13TeV_df['zdPP_cut']
for cut in zdzdPP_23a_df['Cuts']:
    if cut not in zdPP_to_zd13TeV_df['zdPP_cut'].values:
        if '*' not in cut:  # Exclude section headers
            print(cut)

jpsi
upsilon


***
## 4. Text file --> cutlfow string
***

Extract cutflow strings from a text file.

NEED TO IMPLEMENT

### 4.1 ZdZd13TeV cutflows

In [64]:
def parse_cutflow_file(filepath):
    """
    Parse a file containing multiple cutflows.
    
    Parameters:
    -----------
    filepath : str
        Path to the cutflow file
        
    Returns:
    --------
    dict
        Dictionary where keys are cutflow headers (e.g., "zd5_23a --- reco")
        and values are the corresponding cutflow table strings
    """
    with open(filepath, 'r') as f:
        content = f.read()
    
    # Split by lines containing " --- " which mark the start of each cutflow
    lines = content.split('\n')
    
    cutflows = {}
    current_key = None
    current_lines = []
    
    for line in lines:
        # Check if this line is a header (contains " --- ")
        if ' --- ' in line and not line.startswith('|'):
            # Save the previous cutflow if it exists
            if current_key is not None:
                cutflows[current_key] = '\n'.join(current_lines).strip()
            
            # Start a new cutflow
            current_key = line.strip()
            current_lines = []
        else:
            # Add line to current cutflow
            current_lines.append(line)
    
    # Don't forget to save the last cutflow
    if current_key is not None:
        cutflows[current_key] = '\n'.join(current_lines).strip()
    
    return cutflows

# Usage example
# cutflows = parse_cutflow_file('filepath')